# AI Use Case 1 — Pydantic: saída estruturada de agentes (structured output)

**Problema:** um LLM devolve **texto**. O seu código precisa de **dados**: um objeto com campos tipados, valores dentro de um
conjunto fechado, números que batem. Sem contrato, você acaba com regex frágil sobre a resposta e bugs silenciosos.

**Solução:** o modelo Pydantic vira o **contrato** entre o LLM e o código:

```
  modelo Pydantic ──model_json_schema()──► schema enviado ao LLM (instrução + restrição de decodificação)
        ▲                                                │
        │                                                ▼
  objeto tipado ◄──model_validate_json()── resposta JSON do LLM
        │                     │
        │                     └── ValidationError ──► feedback ao LLM ──► nova tentativa (retry)
        ▼
  código de negócio (sem if defensivo)
```

Roteiro:

1. O contrato: `Field(description=...)` é *prompt engineering*
2. O schema que vai para o LLM
3. Chamando provedores reais (código de referência)
4. Parser robusto + **retry com feedback de validação** (para quando não há decodificação restrita)
5. Schema garante **forma**, não **verdade**: validators semânticos contra alucinação
6. Modelando incerteza: `null`, `confidence` e evidência
7. Streaming: validação parcial
8. Checklist

> Todos os exemplos rodam **offline** com `ScriptedLLM` (`fake_llm.py`), que devolve respostas roteirizadas.

In [1]:
import json, re
from datetime import date
from decimal import Decimal
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator, TypeAdapter
from fake_llm import ScriptedLLM

## 1. O contrato

Em structured output, **as descrições dos campos são parte do prompt**: elas chegam ao modelo dentro do JSON Schema.
Escreva-as como instrução — o que é o campo, formato, o que fazer quando a informação não existe.

O exemplo: extrair um lead (contato comercial) de um e-mail livre.

In [2]:
class Lead(BaseModel):
    """Contato comercial extraído de um e-mail recebido."""
    model_config = ConfigDict(extra="forbid")      # vira additionalProperties: false no schema

    name: str = Field(description="Nome completo da pessoa que escreveu o e-mail")
    company: str | None = Field(description="Empresa da pessoa; null se não for mencionada")
    email: str = Field(description="E-mail de contato, exatamente como aparece no texto")
    plan_interest: Literal["free", "pro", "enterprise", "unknown"] = Field(
        description="Plano de interesse. Use 'unknown' se não houver menção clara")
    seats: int | None = Field(ge=1, description="Número de licenças/usuários pedido; null se não informado")
    wants_demo: bool = Field(description="True somente se a pessoa pedir explicitamente uma demonstração")

## 2. O schema que vai para o LLM

In [3]:
schema = Lead.model_json_schema()
print(f"json.dumps(schema, indent=2, ensure_ascii=False):\n{json.dumps(schema, indent=2, ensure_ascii=False)}")

json.dumps(schema, indent=2, ensure_ascii=False):
{
  "additionalProperties": false,
  "description": "Contato comercial extraído de um e-mail recebido.",
  "properties": {
    "name": {
      "description": "Nome completo da pessoa que escreveu o e-mail",
      "title": "Name",
      "type": "string"
    },
    "company": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "description": "Empresa da pessoa; null se não for mencionada",
      "title": "Company"
    },
    "email": {
      "description": "E-mail de contato, exatamente como aparece no texto",
      "title": "Email",
      "type": "string"
    },
    "plan_interest": {
      "description": "Plano de interesse. Use 'unknown' se não houver menção clara",
      "enum": [
        "free",
        "pro",
        "enterprise",
        "unknown"
      ],
      "title": "Plan Interest",
      "type": "string"
    },
    "seats": {
      "anyOf": [
        {

Repare:
* `extra="forbid"` virou `"additionalProperties": false` — exigido por vários provedores no modo estrito.
* `Literal` virou `"enum"` — o modelo só pode escolher entre essas opções (com decodificação restrita).
* Todos os campos estão em `required` (mesmo os que aceitam `null`). Isso é proposital: no modo estrito, os provedores
  costumam exigir que todo campo esteja presente — a "opcionalidade" é expressa com `null`, não com ausência.
* Restrições como `ge=1` aparecem no schema, mas **nem todo provedor as impõe na decodificação**; o SDK pode removê-las
  do schema enviado e validá-las localmente depois. Por isso a validação Pydantic do lado do cliente continua obrigatória.

## 3. Chamando provedores reais (referência, não executado)

**Anthropic (Claude)** — o SDK Python aceita o modelo Pydantic direto e devolve o objeto validado:

```python
from anthropic import Anthropic

client = Anthropic()  # usa ANTHROPIC_API_KEY
response = client.messages.parse(
    model="claude-opus-5-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": f"Extraia o lead deste e-mail:\n\n{email_text}"}],
    output_format=Lead,            # o SDK converte para output_config.format (json_schema)
)
lead: Lead = response.parsed_output
```

Sem o helper, a requisição crua usa `output_config={"format": {"type": "json_schema", "schema": ...}}`.
Para *tool use*, o equivalente é declarar a ferramenta com `"strict": true` (próximo notebook).

**OpenAI** — mesmo padrão:

```python
from openai import OpenAI

client = OpenAI()
response = client.responses.parse(
    model="<modelo>",
    input=[{"role": "user", "content": f"Extraia o lead deste e-mail:\n\n{email_text}"}],
    text_format=Lead,
)
lead: Lead = response.output_parsed
```

Mesmo com decodificação restrita, trate os casos em que **não há objeto**: recusa do modelo, resposta cortada por
`max_tokens` (`stop_reason`), ou validação local falhando em restrições que o provedor não impôs.

Como fica esse tratamento em código — **antes** de tentar validar, olhe o `stop_reason`:

In [4]:
from fake_llm import LLMResponse

class NoStructuredOutput(Exception):
    pass

def parse_response(resp: LLMResponse, model: type[BaseModel]) -> BaseModel:
    match resp.stop_reason:
        case "refusal":
            raise NoStructuredOutput("modelo recusou: não use o conteúdo, responda com mensagem padrão")
        case "max_tokens":
            raise NoStructuredOutput("resposta truncada: aumente max_tokens ou peça um objeto menor (JSON incompleto)")
    try:
        return model.model_validate_json(resp.text)
    except ValidationError as e:                       # restrição que o provedor não impôs (ex.: ge=1)
        raise NoStructuredOutput(f"validação local falhou: {[(x['loc'], x['type']) for x in e.errors()]}") from e

responses = {
    "ok": LLMResponse(text='{"name": "Ana", "company": null, "email": "ana@x.com", "plan_interest": "pro", "seats": 3, "wants_demo": false}'),
    "truncada": LLMResponse(text='{"name": "Ana", "company": nu', stop_reason="max_tokens"),
    "recusa": LLMResponse(text="Não posso ajudar com isso.", stop_reason="refusal"),
    "seats=0": LLMResponse(text='{"name": "Ana", "company": null, "email": "ana@x.com", "plan_interest": "pro", "seats": 0, "wants_demo": false}'),
}
for case, resp in responses.items():
    try:
        print(f"{case}: {parse_response(resp, Lead)!r}")
    except NoStructuredOutput as e:
        print(f"{case}: NoStructuredOutput: {e}")

ok: Lead(name='Ana', company=None, email='ana@x.com', plan_interest='pro', seats=3, wants_demo=False)
truncada: NoStructuredOutput: resposta truncada: aumente max_tokens ou peça um objeto menor (JSON incompleto)
recusa: NoStructuredOutput: modelo recusou: não use o conteúdo, responda com mensagem padrão
seats=0: NoStructuredOutput: validação local falhou: [(('seats',), 'greater_than_equal')]


## 4. Parser robusto + retry com feedback

Nem todo modelo/provedor tem decodificação restrita (modelos locais, gateways, provedores antigos). Aí o LLM pode devolver:
JSON dentro de bloco markdown, texto antes do JSON, campo faltando, tipo errado, valor fora do enum.

A estratégia que funciona na prática:
1. extrair o JSON do texto;
2. validar com Pydantic;
3. se falhar, **devolver os erros do Pydantic ao modelo** e pedir correção — o `ValidationError` é um ótimo feedback, porque diz o campo e o problema;
4. limitar o número de tentativas.

In [5]:
_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)

def extract_json(text: str) -> str:
    """Pega o JSON de dentro de ```json ... ``` ou o primeiro {...} balanceado do texto."""
    if m := _FENCE.search(text):
        return m.group(1).strip()
    start = text.find("{")
    if start == -1:
        raise ValueError("nenhum objeto JSON na resposta")
    depth = 0
    for i, ch in enumerate(text[start:], start):
        depth += ch == "{"
        depth -= ch == "}"
        if depth == 0:
            return text[start:i + 1]
    raise ValueError("JSON incompleto na resposta")

def format_errors(e: ValidationError) -> str:
    return "\n".join(f"- campo `{'.'.join(map(str, err['loc'])) or '(objeto)'}`: {err['msg']} (recebido: {err.get('input')!r})"
                     for err in e.errors(include_url=False))

class StructuredOutputError(Exception):
    pass

def structured_call[T: BaseModel](llm, model: type[T], user_prompt: str, max_attempts: int = 3) -> T:
    system = (
        "Responda SOMENTE com um objeto JSON válido que siga este JSON Schema:\n"
        + json.dumps(model.model_json_schema(), ensure_ascii=False)
    )
    messages = [{"role": "user", "content": user_prompt}]
    for attempt in range(1, max_attempts + 1):
        reply = llm(messages, system=system).text
        try:
            return model.model_validate_json(extract_json(reply))
        except (ValidationError, ValueError) as e:
            feedback = format_errors(e) if isinstance(e, ValidationError) else str(e)
            print(f"  tentativa {attempt} falhou:\n    " + feedback.replace("\n", "\n    "))
            messages += [
                {"role": "assistant", "content": reply},
                {"role": "user", "content": f"Sua resposta não passou na validação:\n{feedback}\n"
                                            "Corrija e responda novamente apenas com o JSON."},
            ]
    raise StructuredOutputError(f"{model.__name__}: sem resposta válida após {max_attempts} tentativas")

In [6]:
email_text = """Oi, aqui é a Marina Costa, da Lumen Analytics. Vimos o plano Enterprise e queremos
licenças para umas 40 pessoas. Conseguem marcar uma demonstração semana que vem? marina.costa@lumen.io"""

llm = ScriptedLLM([
    # 1ª resposta: markdown + plano fora do enum + seats como texto
    'Claro! Aqui está:\n```json\n{"name": "Marina Costa", "company": "Lumen Analytics", '
    '"email": "marina.costa@lumen.io", "plan_interest": "Enterprise", "seats": "uns 40", "wants_demo": true}\n```',
    # 2ª resposta: corrigida após o feedback
    '{"name": "Marina Costa", "company": "Lumen Analytics", "email": "marina.costa@lumen.io", '
    '"plan_interest": "enterprise", "seats": 40, "wants_demo": true}',
])

lead = structured_call(llm, Lead, f"Extraia o lead deste e-mail:\n\n{email_text}")
print(f"\nresultado: {lead}")
print(f"chamadas ao LLM: {len(llm.calls)}")
print(f'\núltima mensagem enviada ao modelo (o feedback):\n{llm.calls[1]["messages"][-1]["content"]}')

  tentativa 1 falhou:
    - campo `plan_interest`: Input should be 'free', 'pro', 'enterprise' or 'unknown' (recebido: 'Enterprise')
    - campo `seats`: Input should be a valid integer, unable to parse string as an integer (recebido: 'uns 40')

resultado: name='Marina Costa' company='Lumen Analytics' email='marina.costa@lumen.io' plan_interest='enterprise' seats=40 wants_demo=True
chamadas ao LLM: 2

última mensagem enviada ao modelo (o feedback):
Sua resposta não passou na validação:
- campo `plan_interest`: Input should be 'free', 'pro', 'enterprise' or 'unknown' (recebido: 'Enterprise')
- campo `seats`: Input should be a valid integer, unable to parse string as an integer (recebido: 'uns 40')
Corrija e responda novamente apenas com o JSON.


## 5. Schema garante forma, não verdade

Decodificação restrita garante que o JSON tem a forma certa. **Não** garante que os números batem, que a data faz sentido ou
que o modelo não inventou um valor. Validators Pydantic adicionam checagens **semânticas** baratas — e o erro volta como feedback.

Caso: extrair uma nota fiscal de texto (OCR). Verificamos: total = soma dos itens, datas coerentes, CNPJ com 14 dígitos.

In [7]:
class InvoiceLine(BaseModel):
    description: str
    quantity: Decimal = Field(gt=0)
    unit_price: Decimal = Field(ge=0)
    line_total: Decimal

    @model_validator(mode="after")
    def check_line(self):
        if abs(self.quantity * self.unit_price - self.line_total) > Decimal("0.01"):
            raise ValueError(f"line_total {self.line_total} != quantity*unit_price "
                             f"({self.quantity * self.unit_price}) em '{self.description}'")
        return self

class Invoice(BaseModel):
    model_config = ConfigDict(extra="forbid")
    supplier_cnpj: str = Field(description="CNPJ do emitente, apenas os 14 dígitos")
    issue_date: date
    due_date: date
    lines: list[InvoiceLine] = Field(min_length=1)
    total: Decimal = Field(description="Valor total da nota como impresso no documento")

    @field_validator("supplier_cnpj")
    @classmethod
    def cnpj_digits(cls, v: str) -> str:
        digits = re.sub(r"\D", "", v)
        if len(digits) != 14:
            raise ValueError("CNPJ deve ter 14 dígitos")
        return digits

    @model_validator(mode="after")
    def check_consistency(self):
        computed = sum((l.line_total for l in self.lines), Decimal("0"))
        if abs(computed - self.total) > Decimal("0.01"):
            raise ValueError(f"total {self.total} não bate com a soma das linhas ({computed}). Releia o documento.")
        if self.due_date < self.issue_date:
            raise ValueError("due_date anterior a issue_date")
        return self

ocr_text = """NF 4471 - Papelaria Central LTDA - CNPJ 12.345.678/0001-90
Emissão: 03/11/2026  Vencimento: 03/12/2026
Resma A4 x10 ....... R$ 32,50 un ..... R$ 325,00
Caneta azul x50 .... R$ 1,90 un ...... R$ 95,00
TOTAL ............................... R$ 420,00"""

llm = ScriptedLLM([
    # alucinação típica: leu 1,90 como 1,99 e "arredondou" o total
    '{"supplier_cnpj": "12.345.678/0001-90", "issue_date": "2026-11-03", "due_date": "2026-12-03", '
    '"lines": [{"description": "Resma A4", "quantity": 10, "unit_price": "32.50", "line_total": "325.00"}, '
    '{"description": "Caneta azul", "quantity": 50, "unit_price": "1.99", "line_total": "95.00"}], "total": "420.00"}',
    '{"supplier_cnpj": "12.345.678/0001-90", "issue_date": "2026-11-03", "due_date": "2026-12-03", '
    '"lines": [{"description": "Resma A4", "quantity": 10, "unit_price": "32.50", "line_total": "325.00"}, '
    '{"description": "Caneta azul", "quantity": 50, "unit_price": "1.90", "line_total": "95.00"}], "total": "420.00"}',
])
inv = structured_call(llm, Invoice, f"Extraia a nota fiscal:\n{ocr_text}")
print(f"\ninv.model_dump_json(indent=2):\n{inv.model_dump_json(indent=2)}")

  tentativa 1 falhou:
    - campo `lines.1`: Value error, line_total 95.00 != quantity*unit_price (99.50) em 'Caneta azul' (recebido: {'description': 'Caneta azul', 'quantity': 50, 'unit_price': '1.99', 'line_total': '95.00'})

inv.model_dump_json(indent=2):
{
  "supplier_cnpj": "12345678000190",
  "issue_date": "2026-11-03",
  "due_date": "2026-12-03",
  "lines": [
    {
      "description": "Resma A4",
      "quantity": "10",
      "unit_price": "32.50",
      "line_total": "325.00"
    },
    {
      "description": "Caneta azul",
      "quantity": "50",
      "unit_price": "1.90",
      "line_total": "95.00"
    }
  ],
  "total": "420.00"
}


O JSON da primeira resposta era **perfeitamente válido em forma** — um schema sozinho teria aceitado `1.99`.
O validator pegou a inconsistência e o feedback apontou exatamente onde reler.

## 6. Modelando incerteza

Um campo obrigatório **força** o modelo a preencher algo — e ele inventa. Dê saídas honestas:
* `X | None` com instrução "null se não houver" (seção 1);
* um valor `"unknown"` no enum;
* `confidence` + `evidence` (trecho literal do texto) — e **verifique** que a evidência existe no texto de origem.

In [8]:
class ExtractedFact(BaseModel):
    value: str | None = Field(description="Valor extraído; null se o texto não informa")
    evidence: str | None = Field(description="Trecho LITERAL do texto que comprova o valor; null se value for null")
    confidence: float = Field(ge=0, le=1)

class ContractTerms(BaseModel):
    termination_notice_days: ExtractedFact
    governing_law: ExtractedFact

def verify_evidence(terms: ContractTerms, source: str) -> list[str]:
    problems = []
    for name, fact in terms:
        if fact.value is not None and (not fact.evidence or fact.evidence not in source):
            problems.append(f"{name}: evidência não encontrada no texto -> possível alucinação")
    return problems

contract = "Qualquer parte pode rescindir mediante aviso prévio de 30 (trinta) dias."
llm = ScriptedLLM(['{"termination_notice_days": {"value": "30", "evidence": "aviso prévio de 30 (trinta) dias", "confidence": 0.95},'
                   ' "governing_law": {"value": "Lei brasileira", "evidence": "regido pelas leis do Brasil", "confidence": 0.6}}'])
terms = structured_call(llm, ContractTerms, contract)
print(f"terms.termination_notice_days: {terms.termination_notice_days}")
print(f"verify_evidence(terms, contract): {verify_evidence(terms, contract)}")

terms.termination_notice_days: value='30' evidence='aviso prévio de 30 (trinta) dias' confidence=0.95
verify_evidence(terms, contract): ['governing_law: evidência não encontrada no texto -> possível alucinação']


## 7. Streaming: validação parcial

Com streaming, o JSON chega em pedaços. Para mostrar progresso na UI (ex.: itens de uma lista aparecendo um a um),
valide o prefixo com `experimental_allow_partial` — o último item incompleto é descartado.

⚠️ Limitação medida (Pydantic 2.13): o modo parcial funciona para `list`, `dict` e `TypedDict`, mas **não se propaga para
dentro dos campos de um `BaseModel`**. Se o objeto externo for um `BaseModel`, um item incompleto no meio do stream gera erro.
Padrão que funciona: um **espelho `TypedDict`** (campos `NotRequired`) só para a visão parcial, e o `BaseModel` completo
para a validação final.

In [9]:
from typing import NotRequired, TypedDict

class Task(BaseModel):
    title: str
    owner: str

class Plan(BaseModel):                 # contrato final, estrito
    goal: str
    tasks: list[Task] = Field(min_length=1)

class PlanStream(TypedDict):           # visão parcial para a UI
    goal: NotRequired[str]
    tasks: NotRequired[list[Task]]

PlanStreamAdapter = TypeAdapter(PlanStream)
full = ('{"goal": "Lançar v2", "tasks": [{"title": "Migrar banco", "owner": "ana"}, '
        '{"title": "Revisar API", "owner": "bia"}, {"title": "Escrever release notes", "owner": "caio"}]}')

for cut in (15, 40, 75, 110, 130, len(full)):
    partial = PlanStreamAdapter.validate_json(full[:cut], experimental_allow_partial=True)
    print(f"{cut:>3} chars | goal={partial.get('goal', '…')!r:<12} tarefas: {[t.title for t in partial.get('tasks', [])]}")

plan = Plan.model_validate_json(full)          # fim do stream: validação completa
print(f"final: {plan.tasks[-1]}")

try:                                            # o mesmo prefixo no BaseModel falha
    TypeAdapter(Plan).validate_json(full[:110], experimental_allow_partial=True)
except ValidationError as e:
    print(f'BaseModel + partial no meio de um item: {e.errors(include_url=False)[0]["loc"]}')
    print(f'e.errors(include_url=False)[0]["type"]: {e.errors(include_url=False)[0]["type"]}')

 15 chars | goal='…'          tarefas: []
 40 chars | goal='Lançar v2'  tarefas: []
 75 chars | goal='Lançar v2'  tarefas: ['Migrar banco']
110 chars | goal='Lançar v2'  tarefas: ['Migrar banco']
130 chars | goal='Lançar v2'  tarefas: ['Migrar banco', 'Revisar API']
171 chars | goal='Lançar v2'  tarefas: ['Migrar banco', 'Revisar API', 'Escrever release notes']
final: title='Escrever release notes' owner='caio'
BaseModel + partial no meio de um item: ('tasks', 1, 'owner')
e.errors(include_url=False)[0]["type"]: missing


## ⚠️ Pegadinhas, efeitos colaterais e caveats

### Já demonstradas acima

| # | pegadinha | onde | como evitar |
|---|---|---|---|
| 1 | restrições (`ge`, `pattern`) podem não ser impostas pelo provedor | §2 | validar localmente sempre |
| 2 | resposta com texto em volta, JSON quebrado ou campo faltando | §4 | extração + validação + retry com feedback |
| 3 | schema garante forma, não verdade (soma errada, data inventada) | §5 | validadores semânticos + evidência literal |
| 4 | modo parcial não se propaga para modelos aninhados | §7 | `TypedDict` para a visão parcial |

### Novas

**1. Modelo aninhado sem `extra="forbid"` e campo com default quebram o modo estrito.** O `extra="forbid"` do modelo de fora
**não** se propaga para o schema dos aninhados, e um campo com default sai de `required`. Um verificador simples pega os dois:

In [10]:
class LineItem(BaseModel):                       # esqueceu o extra="forbid"
    sku: str
    qty: int


class Order(BaseModel):
    model_config = ConfigDict(extra="forbid")
    customer: str
    items: list[LineItem]
    notes: str = ""                              # default: deixa de ser obrigatório no schema


def strict_mode_issues(schema: dict) -> list[str]:
    issues = []
    for name, node in [("<raiz>", schema), *schema.get("$defs", {}).items()]:
        if node.get("type") != "object":
            continue
        if node.get("additionalProperties") is not False:
            issues.append(f"{name}: falta additionalProperties: false")
        missing = sorted(set(node.get("properties", {})) - set(node.get("required", [])))
        if missing:
            issues.append(f"{name}: fora de required: {missing}")
    return issues


for issue in strict_mode_issues(Order.model_json_schema()):
    print(f"problema: {issue}")
print(f"Lead (do §1) está ok: {strict_mode_issues(Lead.model_json_schema()) == []}")

problema: <raiz>: fora de required: ['notes']
problema: LineItem: falta additionalProperties: false
Lead (do §1) está ok: True


O provedor recusa o schema (ou, pior, aceita e não impõe). Rode um verificador desses nos testes, para todo modelo que vai a um LLM.
Note também os `$defs`/`$ref`: cada modelo aninhado vira uma referência, e nem todo provedor ou modo aceita todas as formas.

**2. A docstring do modelo vai para o prompt.**

In [11]:
class Ticket(BaseModel):
    """Ticket de suporte. TODO(ana): remover o campo legacy_id depois da migração do Zendesk (custo: US$ 40k/mês)."""
    model_config = ConfigDict(extra="forbid")
    subject: str


print(f"Ticket.model_json_schema()['description']: {Ticket.model_json_schema()['description']}")

Ticket.model_json_schema()['description']: Ticket de suporte. TODO(ana): remover o campo legacy_id depois da migração do Zendesk (custo: US$ 40k/mês).


Comentário interno, nome de pessoa, valor de contrato: tudo isso é enviado ao provedor **em toda chamada** (e custa tokens). A
docstring de um modelo de structured output é instrução para o LLM; notas internas vão em comentários `#`.

**3. A ordem dos campos é a ordem de geração.** O modelo escreve o JSON na ordem do schema. Pedir a resposta **antes** da
justificativa faz o modelo decidir primeiro e justificar depois:

In [12]:
class DecisionFirst(BaseModel):
    approve: bool
    reasoning: str


class ReasoningFirst(BaseModel):
    reasoning: str = Field(description="Analise os critérios antes de decidir")
    approve: bool


print(f"ordem em DecisionFirst: {list(DecisionFirst.model_json_schema()['properties'])}")
print(f"ordem em ReasoningFirst: {list(ReasoningFirst.model_json_schema()['properties'])}")

ordem em DecisionFirst: ['approve', 'reasoning']
ordem em ReasoningFirst: ['reasoning', 'approve']


Mudar a ordem dos campos de um modelo "só por organização" muda o comportamento do LLM. Trate a ordem como parte do contrato (e
coloque evidências/raciocínio antes da conclusão).

**4. `Literal` diferencia maiúsculas: `"Pro"` não é `"pro"`.**

In [13]:
from typing import Annotated
from pydantic import BeforeValidator

Plan = Literal["free", "pro", "enterprise", "unknown"]
NormalizedPlan = Annotated[Plan, BeforeValidator(lambda v: v.strip().lower() if isinstance(v, str) else v)]

for plan_type, label in [(Plan, "Literal puro"), (NormalizedPlan, "normalizado")]:
    try:
        print(f"{label}: {TypeAdapter(plan_type).validate_python(' Pro ')!r}")
    except ValidationError as ex:
        print(f"{label}: {ex.errors()[0]['msg']}")

Literal puro: Input should be 'free', 'pro', 'enterprise' or 'unknown'
normalizado: 'pro'


Sem decodificação restrita, variações de caixa e espaços são frequentes — e cada uma custa um retry. Normalizar na entrada é barato;
só não normalize o que tem significado (um código de produto `ABC` não é `abc`).

## 8. Checklist

| ✔ | prática |
|---|---|
| ☐ | Um modelo Pydantic por tarefa de extração; descrições dos campos escritas como instrução |
| ☐ | `extra="forbid"` e campos sempre presentes (opcionalidade via `null`) para compatibilidade com modo estrito |
| ☐ | `Literal`/`Enum` para todo campo categórico — inclua uma opção "unknown"/"other" |
| ☐ | Use decodificação restrita do provedor quando existir (`messages.parse`, `responses.parse`) |
| ☐ | Mesmo assim: **valide localmente** e trate recusa, `max_tokens` e restrições não impostas |
| ☐ | Sem decodificação restrita: extrair JSON + validar + retry com `ValidationError` como feedback (limite de tentativas) |
| ☐ | Validators semânticos (somas, datas, formatos) para pegar alucinação que o schema não pega |
| ☐ | Evidência literal + checagem contra o texto fonte em extrações críticas |
| ☐ | Logue tentativas e erros de validação: são o melhor dado para melhorar prompt e schema |

➡️ Próximo: [`02_pydantic_tool_calling.ipynb`](02_pydantic_tool_calling.ipynb)